# Chapter 19: Web Scraping in Parallel

In [1]:
import random
import re
import threading
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

## MultiThreaded Crawling

In [2]:
def print_time(threadName, delay, iterations):
    start = int(time.time())
    for i in range(iterations):
        time.sleep(delay)
        print(f"{int(time.time() - start)} -- {threadName}")

In [3]:
threads = [
    threading.Thread(target=print_time, args=("Fizz", 1, 33)),
    threading.Thread(target=print_time, args=("Buzz", 3, 20)),
    threading.Thread(target=print_time, args=("Counter", 0.5, 55)),
]

print([t.start() for t in threads])
print()
print([t.join() for t in threads])

[None, None, None]

0 -- Counter
1 -- Fizz
1 -- Counter
1 -- Counter
2 -- Fizz
2 -- Counter
2 -- Counter
3 -- Buzz
3 -- Fizz
3 -- Counter
3 -- Counter
4 -- Fizz
4 -- Counter
4 -- Counter
5 -- Fizz
5 -- Counter
5 -- Counter
6 -- Buzz
6 -- Fizz
6 -- Counter
6 -- Counter
7 -- Fizz
7 -- Counter
7 -- Counter
8 -- Fizz
8 -- Counter
8 -- Counter
9 -- Buzz
9 -- Fizz
9 -- Counter
9 -- Counter
10 -- Fizz
10 -- Counter
10 -- Counter
11 -- Fizz
11 -- Counter
11 -- Counter
12 -- Buzz
12 -- Fizz
12 -- Counter
12 -- Counter
13 -- Fizz
13 -- Counter
13 -- Counter
14 -- Fizz
14 -- Counter
14 -- Counter
15 -- Buzz
15 -- Fizz
15 -- Counter
15 -- Counter
16 -- Fizz
16 -- Counter
16 -- Counter
17 -- Fizz
17 -- Counter
17 -- Counter
18 -- Buzz
18 -- Fizz
18 -- Counter
18 -- Counter
19 -- Fizz
19 -- Counter
19 -- Counter
20 -- Fizz
20 -- Counter
20 -- Counter
21 -- Buzz
21 -- Fizz
21 -- Counter
21 -- Counter
22 -- Fizz
22 -- Counter
22 -- Counter
23 -- Fizz
23 -- Counter
23 -- Counter
24 -- Buzz
24 -- Fizz
2

In [9]:
WIKI_SCRAPED_URLS = 0
WIKI_VISITED = []

def get_links(thread_name, bs):
    print(f"Gettings links in {thread_name}")
    links = bs.find("div", {"id": "bodyContent"}).find_all(
        "a",
        href = re.compile(r"^(?:https://en\.wikipedia\.org)?/wiki/[^:#]+$"),
    )
    return [link for link in links if link not in WIKI_VISITED]

def scrape_article(thread_name, path):
    """Recursively find links on Wikipedia page,
    then follow a random link, with artficial 3 sec delay.
    """
    global WIKI_SCRAPED_URLS
    global WIKI_VISITED

    WIKI_VISITED.append(path)
    print(f"{thread_name}: Scraping {path!r} ...")
    # time.sleep(3)
    resp = requests.get(
        urljoin("https://en.wikipedia.org", path),
        headers={"User-Agent": "Mr.Robot-Linux_Machine"}
    )
    bs = BeautifulSoup(resp.text)
    title = bs.find("h1").get_text(strip=True)
    print("\tTitle:", title)
    # links = bs.find("div", {"id": "bodyContent"}).find_all(
    #     "a",
    #     # href=re.compile("^(/wiki/)((?!:).)*$"),
    #     href = re.compile(r"^(?:https://en\.wikipedia\.org)?/wiki/[^:#]+$"),
    # )
    links = get_links(thread_name, bs)

    if WIKI_SCRAPED_URLS >= 100:  # stop if the scraping limit hits
        return
    if len(links) > 0:
        newArticle = links[random.randint(0, len(links) - 1)].attrs["href"]
        WIKI_SCRAPED_URLS += 1
        scrape_article(thread_name, newArticle)
    else:
        print("No links found.")

In [8]:
threads = [
    threading.Thread(target=scrape_article, args=("Thread 1", "/wiki/Kevin_Bacon")),
    threading.Thread(target=scrape_article, args=("Thread 2", "/wiki/Monty_Python")),
]

print([t.start() for t in threads])
print([t.join() for t in threads])

Thread 1: Scraping '/wiki/Kevin_Bacon' ...
Thread 2: Scraping '/wiki/Monty_Python' ...
[None, None]
	Title: Kevin Bacon
Gettings links in Thread 1
Thread 1: Scraping 'https://en.wikipedia.org/wiki/Social_networking_service' ...
	Title: Monty Python
Gettings links in Thread 2
Thread 2: Scraping 'https://en.wikipedia.org/wiki/Sit_on_My_Face' ...
	Title: Social networking service
Gettings links in Thread 1
Thread 1: Scraping 'https://en.wikipedia.org/wiki/The_Journal_(Edinburgh)' ...
	Title: Sit on My Face
Gettings links in Thread 2
Thread 2: Scraping 'https://en.wikipedia.org/wiki/And_Now_for_Something_Completely_Different' ...
	Title: The Journal(student newspaper)
Gettings links in Thread 1
Thread 1: Scraping 'https://en.wikipedia.org/wiki/MRC_Human_Genetics_Unit' ...
	Title: MRC Human Genetics Unit
Gettings links in Thread 1
Thread 1: Scraping 'https://en.wikipedia.org/wiki/Pleasantine_Mill' ...
	Title: And Now for Something Completely Different
Gettings links in Thread 2
Thread 2: Sc

### Race Conditions and Queues